# Splitting the clean corpus

This notebook shows how `split_corpus.py` splits the clean corpus, and how many
verses each lot holds. It shows counts only, never verses, so that the
validation and test sets stay unseen.

**Method.**
- The split is made **by poem**: the verses of a poem share poet, rhyme and
  style, so a poem is never on both sides.
- It is made **baḥr by baḥr**, so that every baḥr has the same proportions in
  each lot.
- Within a baḥr, the poems are ordered by a **hash of their identifier** (seed
  2026), not by a global shuffle: a poem keeps its place even if other poems are
  removed later.
- The poems used to tune the first prototype of f (the 160 verses of
  `arud/prototype/evaluate.py`, 154 poems) come first, so they are in C.

**Lots.**
- **C** (75 %): the development pool. f is written on growing tiers of C. What
  is left of C when f is frozen (B) is kept for the network of f\*, and is never
  looked at while f is written.
- **validation** (10 %): to check the rules of f on unseen verses, and to tune
  the networks.
- **test** (15 %): used once, at the end.

**Observed** marks what the data show, **Interpretation** our own reading.

In [1]:
import pandas as pd
from IPython import get_ipython

# Display every table with a thousands separator (72,588).
html = get_ipython().display_formatter.formatters["text/html"]
html.for_type(pd.DataFrame, lambda df: df.style.format(thousands=",", precision=1).to_html())
html.for_type(pd.Series, lambda s: s.to_frame().style.format(thousands=",", precision=1).to_html())

In [2]:
from split_corpus import RARE, SEEN_POEMS, TIERS, development_pool, load_split

corpus = load_split()
LOTS = ["C", "validation", "test"]
lots = corpus["split"].value_counts().reindex(LOTS)
pd.DataFrame({"verses": lots, "%": (lots / len(corpus) * 100).round(1)})

,verses,%
split,,
C,"198,828",74.9
validation,"26,458",10.0
test,"40,095",15.1


## A and B within C

C is not split into A and B in advance. **A** is the part of C that f is
written on: it grows tier by tier, and is **at most 60 % of C** (the last tier,
`TIERS[-1]`). **B** is what is left of C when f is frozen: it is **at least
40 % of C**, and is never looked at while f is written. It is kept for the
network of f\*. If f is frozen at an earlier tier, A is smaller and B larger.

C, validation and test make up the corpus (75 + 10 + 15 = 100 %). A and B do not
add to C: they divide it. The table gives A and B at the limit of 60 %, for the
whole corpus (variants and rare buḥūr included).

In [3]:
n, n_C = len(corpus), (corpus["split"] == "C").sum()
n_A = len(development_pool(corpus, TIERS[-1]))
rows = {
    "A (at most 60 % of C)": n_A,
    "B (at least 40 % of C)": n_C - n_A,
    "C = A + B": n_C,
    "validation": lots["validation"],
    "test": lots["test"],
    "total": n,
}
pd.DataFrame({"verses": rows,
              "% of corpus": {k: round(v / n * 100, 1) for k, v in rows.items()},
              "% of C": {k: round(v / n_C * 100, 1) for k, v in list(rows.items())[:3]}}).fillna("")

,verses,% of corpus,% of C
A (at most 60 % of C),"119,274",44.9,60.0
B (at least 40 % of C),"79,554",30.0,40.0
C = A + B,"198,828",74.9,100.0
validation,"26,458",10.0,
test,"40,095",15.1,
total,"265,381",100.0,


## Verses per baḥr and lot

Unit: **verses**, without the variants (مجزوء, ...), as given by `filter_bahr`.
The variants are split in the same way.

In [4]:
main = corpus[corpus["variant"].isna()]
table = pd.crosstab(main["bahr"], main["split"]).reindex(columns=LOTS)
table["total"] = table.sum(axis=1)
table = table.sort_values("total", ascending=False)
table.loc["total"] = table.sum()
table

split,C,validation,test,total
bahr,,,,
طويل,"54,431","7,248","10,909","72,588"
كامل,"43,492","5,887","8,667","58,046"
بسيط,"24,904","3,280","4,885","33,069"
وافر,"17,182","2,344","3,437","22,963"
خفيف,"14,851","1,953","2,926","19,730"
متقارب,"8,402","1,062","1,751","11,215"
رجز,"8,068","1,039","1,598","10,705"
سريع,"5,769",751,"1,185","7,705"
رمل,"4,683",509,"1,112","6,304"


**Observed.** Every baḥr has about 75 / 10 / 15 % of its verses in C,
validation and test. The small gaps come from the cut between two poems.

## The rare buḥūr

For المتدارك, المقتضب and المضارع, there are too few poems for a lot B. They have
no f\*: f\* returns the answer of f. Their split is made on the **number of
poems** (70 / 15 / 15), and their part of C is available from the first tier.

In [5]:
rare = corpus[corpus["bahr"].isin(RARE)]
poems = pd.crosstab(rare["bahr"], rare["split"], values=rare["poem_id"], aggfunc="nunique").reindex(columns=LOTS)
verses = pd.crosstab(rare["bahr"], rare["split"]).reindex(columns=LOTS)
pd.concat({"poems": poems, "verses": verses}, axis=1)

poems                 verses                
split      C validation test      C validation test
bahr                                               
متدارك    41          9    9    306        125   99
مضارع      2          1    1      5          2    3
مقتضب     14          3    3    118         42    8

**Interpretation.** These lots are small: a test of a few poems gives only an
indicative score. Their false alarms (f announcing one of them by mistake) can
be measured on the much larger test of the other buḥūr.

## Development tiers

f is written tier by tier: first on the first 0.5 % of C, then 1 %, 2 %, ...
Each tier contains the previous one and takes the same share of every baḥr.
0.60 is the limit: what is left of C (at least 40 % of each baḥr) becomes B.

Unit: **verses** available at each tier, without the variants. The rare
buḥūr are left out of this table: their whole part of C is available from the
first tier (متدارك 268 verses, مقتضب 118, مضارع 5).

In [6]:
columns = {}
for tier in TIERS:
    pool = development_pool(corpus, tier)
    columns[f"{tier * 100:g} %"] = pool[pool["variant"].isna()].groupby("bahr").size()
tiers = pd.DataFrame(columns).fillna(0).astype(int).reindex(table.index.drop("total"))
tiers = tiers.drop(index=sorted(RARE))
tiers.loc["total"] = tiers.sum()
tiers

,0.5 %,1 %,2 %,5 %,10 %,20 %,30 %,40 %,50 %,60 %
bahr,,,,,,,,,,
طويل,237,544,"1,086","2,718","5,444","10,865","16,325","21,776","27,218","32,642"
كامل,212,441,859,"1,099","4,541","8,779","13,111","17,466","21,653","26,137"
بسيط,90,251,356,"1,218","2,558","5,025","7,543","9,984","12,480","14,939"
وافر,78,148,325,844,"1,663","3,379","5,110","6,857","8,591","10,327"
خفيف,70,140,297,732,"1,477","2,957","4,483","5,964","7,442","8,934"
متقارب,21,52,169,419,841,"1,684","2,509","3,348","4,183","5,033"
رجز,13,66,154,377,509,"1,628","2,483","3,321","3,983","4,773"
سريع,27,55,115,276,557,"1,143","1,708","2,263","2,869","3,446"
رمل,15,42,69,202,437,892,"1,380","1,872","2,312","2,790"


**Observed.** The first tier holds a few hundred verses of the large buḥūr
(237 of ṭawīl). The smallest buḥūr only appear from 2 % (مجتث, هزج). Some steps
jump because of long poems.

In [7]:
seen = corpus[corpus["poem_id"].isin(SEEN_POEMS)]
print(f"{seen['poem_id'].nunique()} poems of evaluate.py in the corpus, lots: {sorted(seen['split'].unique())}")

153 poems of evaluate.py in the corpus, lots: ['C']
